# A t-SNE map of three pumps

A plant logs six vibration features from each of three pumps, 40 readings per pump. One
more reading came in from pump 0 while a sensor was disconnected, and the logger wrote
9999 in place of the missing value. Nobody has removed it. You will build t-SNE in NumPy
and map all 121 readings: the per-point bandwidths, the pair probabilities, the KL cost
and its gradient, and the gradient descent that places the points.

**How to work through it.** Each part asks for one or two functions. Replace the `TODO`
and the `raise NotImplementedError` with your code, then run the check cell under it. The
check prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

rng = np.random.default_rng(23)
pump_means = 4.0 * rng.normal(size=(3, 6))         # each pump's typical reading
X = np.vstack([c + rng.normal(size=(40, 6)) for c in pump_means])
pump = np.repeat([0, 1, 2], 40)
glitch = X[7].copy()
glitch[2] = 9999.0                                # the logger's placeholder for a missing value
X = np.vstack([X, glitch])
pump = np.r_[pump, 0]
D2 = ((X[:, None, :] - X[None, :, :]) ** 2).sum(-1)   # squared distances, 121 x 121
print(f"{len(X)} readings, the last one is the glitch; its smallest squared distance is {D2[-1, :-1].min():.3g}")

## Part 1 (coding task): A bandwidth for every reading

Write `conditional_probs(D2, perplexity)`. For each point $i$, find the bandwidth
$\sigma_i$ at which the neighbor distribution

$$
p_{j \mid i} = \frac{\exp(-D2_{ij} / 2\sigma_i^2)}{\sum_{k \ne i} \exp(-D2_{ik} / 2\sigma_i^2)}
$$

has the given perplexity, $2^{H}$ with $H$ the entropy in bits (or, equivalently, $e^{H}$
with $H$ in nats). Perplexity rises with $\sigma_i$, so bisection finds it: search between
`1e-3` and `1e5` for 64 steps. Return the $121 \times 121$ matrix whose row $i$ holds the
$p_{j \mid i}$, with zeros on the diagonal, and the array of bandwidths.

Every row has to reach the target, the glitch reading's row included. If a row's squared
distances are all huge, subtract the row's smallest one first: it multiplies every weight by
the same constant and cancels when you normalize.

In [ ]:
def conditional_probs(D2, perplexity, steps=64):
    # TODO
    raise NotImplementedError

In [ ]:
def row_perplexities(P):
    out = np.empty(len(P))
    for i, row in enumerate(P):
        r = row[row > 0]
        out[i] = np.exp(-(r * np.log(r)).sum())
    return out

P_cond, sigmas = conditional_probs(D2, 10.0)
P_cond, sigmas = np.asarray(P_cond, dtype=float), np.asarray(sigmas, dtype=float)
assert P_cond.shape == (121, 121) and sigmas.shape == (121,), "return the 121 x 121 matrix and 121 bandwidths"
bad = np.flatnonzero(~np.isfinite(P_cond).all(axis=1))
assert len(bad) == 0, f"rows {bad[:5]} contain nan or inf values"
assert np.allclose(np.diag(P_cond), 0), "a point is not its own neighbor, so the diagonal must be 0"
assert np.allclose(P_cond.sum(axis=1), 1), "each row is a distribution over the other points and must sum to 1"
perp = row_perplexities(P_cond)
off = np.flatnonzero(np.abs(perp - 10) > 1e-3)
assert len(off) == 0, f"rows {off[:5]} have perplexity {np.round(perp[off[:5]], 3)}, not 10"
for i in (0, 45, 100):
    w = np.exp(-np.delete(D2[i], i) / (2 * sigmas[i] ** 2))
    assert np.allclose(np.delete(P_cond[i], i), w / w.sum()), f"row {i} does not match its own bandwidth {sigmas[i]:.4f}"
print(f"bandwidths: median {np.median(sigmas[:-1]):.3f} for the pump readings, {sigmas[-1]:.1f} for the glitch")
print("Part 1 passed")

## Part 2 (coding task): Pair probabilities, the cost and its gradient

Write two functions.

- `joint_probs(P_cond)` returns $p_{ij} = (p_{j \mid i} + p_{i \mid j}) / 2n$.
- `kl_and_grad(P, Y)` takes the joint probabilities and an $n \times 2$ layout `Y`, and
  returns the cost $C = \sum_{i \ne j} p_{ij} \log (p_{ij} / q_{ij})$ (in nats, skipping
  pairs with $p_{ij} = 0$) and its gradient, an $n \times 2$ array with row $i$ equal to

$$
\frac{\partial C}{\partial y_i} = 4 \sum_{j \ne i} (p_{ij} - q_{ij})\, w_{ij}\, (y_i - y_j),
\qquad w_{ij} = \frac{1}{1 + \lVert y_i - y_j \rVert^2},
\qquad q_{ij} = \frac{w_{ij}}{\sum_{k \ne l} w_{kl}} .
$$

In [ ]:
def joint_probs(P_cond):
    # TODO
    raise NotImplementedError


def kl_and_grad(P, Y):
    # TODO
    raise NotImplementedError

In [ ]:
small = np.array([[0.0, 0.7, 0.3], [0.5, 0.0, 0.5], [0.9, 0.1, 0.0]])
J = np.asarray(joint_probs(small), dtype=float)
assert np.allclose(J, [[0, 0.2, 0.2], [0.2, 0, 0.1], [0.2, 0.1, 0]]), f"joint_probs of a small example gave\n{np.round(J, 4)}"

r = np.random.default_rng(5)
A = r.random((6, 6)) * (1 - np.eye(6))
P6 = (A + A.T) / (A + A.T).sum()
Y6 = r.normal(size=(6, 2))
C6, G6 = kl_and_grad(P6, Y6)
W6 = 1 / (1 + ((Y6[:, None] - Y6[None]) ** 2).sum(-1))
np.fill_diagonal(W6, 0)
Q6 = W6 / W6.sum()
off = ~np.eye(6, dtype=bool)
assert np.isclose(C6, (P6[off] * np.log(P6[off] / Q6[off])).sum()), "the cost is wrong: q is normalized over all pairs, not row by row"
G6 = np.asarray(G6, dtype=float)
assert G6.shape == (6, 2), "the gradient has one row per point and one column per output dimension"
num = np.zeros_like(Y6)
for i in range(6):
    for k in range(2):
        e = np.zeros_like(Y6)
        e[i, k] = 1e-6
        num[i, k] = (kl_and_grad(P6, Y6 + e)[0] - kl_and_grad(P6, Y6 - e)[0]) / 2e-6
assert np.allclose(G6, num, atol=1e-6), f"the gradient does not match finite differences of the cost: largest gap {np.abs(G6 - num).max():.2e}"

P = np.asarray(joint_probs(P_cond), dtype=float)
assert np.allclose(P, P.T) and np.isclose(P.sum(), 1), "the joint probabilities must be symmetric and sum to 1"
print(f"the glitch's total probability: {P[-1].sum():.6f} (1/2n = {1 / (2 * len(P)):.6f})")
print("Part 2 passed")

## Part 3 (coding task): Place the points

Write `embed(P, n_iter=500, lr=50.0, seed=0)`. Start from
`Y = 1e-4 * np.random.default_rng(seed).normal(size=(n, 2))` and a velocity `V` of zeros.
At step $t$:

- for $t < 100$, compute the gradient with every $p_{ij}$ multiplied by 12 and use
  momentum 0.5. From $t = 100$ on, use the plain $p_{ij}$ and momentum 0.8.
- update `V = momentum * V - lr * G` and then `Y = Y + V`.

Return the final `Y`. The check scores the map without plotting it: how many of each
reading's five nearest neighbors in the plane come from its own pump, and where the glitch
lands.

In [ ]:
def embed(P, n_iter=500, lr=50.0, seed=0):
    # TODO
    raise NotImplementedError

In [ ]:
tiny = np.full((4, 4), 0.05)
tiny[0, 1] = tiny[1, 0] = 0.3
np.fill_diagonal(tiny, 0.0)
Y4 = np.asarray(embed(tiny / tiny.sum(), n_iter=200), dtype=float)
assert Y4.shape == (4, 2), "embed should return one row of two coordinates per point"
Y = np.asarray(embed(P), dtype=float)
assert Y.shape == (121, 2) and np.all(np.isfinite(Y)), "embed should return a finite 121 x 2 layout"
C, _ = kl_and_grad(P, Y)
Dy = ((Y[:, None] - Y[None]) ** 2).sum(-1)
np.fill_diagonal(Dy, np.inf)
nn5 = np.argsort(Dy[:120, :120], axis=1)[:, :5]
same = (pump[nn5] == pump[:120, None]).mean()
glitch_nn = int(np.argmin(Dy[-1]))
data_nn = np.argsort(D2[-1, :-1])[:3]
print(f"final KL {C:.3f}, five nearest neighbors from the same pump {same:.1%}")
print(f"the glitch lands next to reading {glitch_nn}; its nearest readings in the data are {data_nn}")
assert C < 1.0, f"the final KL is {C:.2f}, and a correct run ends well below 1"
assert same >= 0.95, f"only {same:.1%} of the five nearest neighbors come from the same pump"
assert glitch_nn in data_nn, "the glitch should land next to one of its three nearest readings in the data"
print("Part 3 passed")

## Solution

### Part 1: A bandwidth for every reading

In [ ]:
def conditional_probs(D2, perplexity, steps=64):
    n = len(D2)
    P = np.zeros((n, n))
    sigmas = np.zeros(n)
    target = np.log(perplexity)                   # compare entropies in nats
    for i in range(n):
        d = np.delete(D2[i], i)
        d = d - d.min()                           # the same shift for every candidate cancels in the ratio
        lo, hi = 1e-3, 1e5
        for _ in range(steps):
            s = np.sqrt(lo * hi)                  # bisect on a log scale
            w = np.exp(-d / (2 * s * s))
            p = w / w.sum()
            nz = p > 0                            # 0 log 0 counts as 0
            if -(p[nz] * np.log(p[nz])).sum() > target:
                hi = s
            else:
                lo = s
        s = np.sqrt(lo * hi)
        w = np.exp(-d / (2 * s * s))
        P[i] = np.insert(w / w.sum(), i, 0.0)
        sigmas[i] = s
    return P, sigmas

**Why it works.** Adding the same constant to every squared distance in row $i$ multiplies every
weight by the same factor, which the normalization divides out. Subtracting the row's
smallest distance therefore changes nothing in exact arithmetic, but it keeps the nearest
weight at exactly 1. Without it the glitch's squared distances, all near $10^8$, make
every weight underflow to 0 at the bandwidth it needs (about 64), and the row becomes
0 / 0. The other rows meet the opposite problem: the glitch's weight in them is exactly 0,
and $0 \cdot \log 0$ must be counted as 0, not computed as nan. Bisecting on a log scale
handles bandwidths from 0.9 to 64 in the same 64 steps.

### Part 2: Pair probabilities, the cost and its gradient

In [ ]:
def joint_probs(P_cond):
    return (P_cond + P_cond.T) / (2 * len(P_cond))


def kl_and_grad(P, Y):
    diff = Y[:, None, :] - Y[None, :, :]           # y_i - y_j, n x n x 2
    W = 1.0 / (1.0 + (diff ** 2).sum(-1))
    np.fill_diagonal(W, 0.0)
    Q = W / W.sum()                               # one normalization over all pairs
    m = P > 0
    C = float((P[m] * np.log(P[m] / Q[m])).sum())
    G = 4 * (((P - Q) * W)[:, :, None] * diff).sum(axis=1)
    return C, G

**Why it works.** Symmetrizing keeps every point's total probability at least $1/2n$, because its own row of
conditionals always sums to 1. The glitch shows it: almost no reading has it as a neighbor,
yet its total is the floor $1/2n$, since its weight in every other row
underflows to 0. In the cost, $q$ comes from one normalizing sum
over every pair, which is what couples all the points and makes distant pairs repel. The
gradient is the derivative of $-\sum p \log w + \log \sum w$, and the finite difference
check confirms the factor 4 and the extra $w_{ij}$.

### Part 3: Place the points

In [ ]:
def embed(P, n_iter=500, lr=50.0, seed=0):
    Y = 1e-4 * np.random.default_rng(seed).normal(size=(len(P), 2))
    V = np.zeros_like(Y)
    for t in range(n_iter):
        exaggeration, momentum = (12.0, 0.5) if t < 100 else (1.0, 0.8)
        _, G = kl_and_grad(exaggeration * P, Y)
        V = momentum * V - lr * G
        Y = Y + V
    return Y

**Why it works.** The momentum step is gradient descent with a running velocity, so the points keep moving
through flat stretches of the cost. Multiplying $P$ by 12 for the first 100 steps makes
almost every pair with a real $p_{ij}$ attract, so each pump's readings gather into a tight
group early, and switching it off lets the groups spread to their final size. Left on, the
cost of the final map stays around 2. The map ends with the pump readings' five nearest neighbors coming from their own
pump at least 95% of the time, and a KL of about 0.47. The glitch, 9999 units from everything,
sits next to reading 74, its nearest reading in the data. Its conditional row puts almost
all its weight on a few readings, and symmetrization keeps that weight, so it stays tied to
them. The one column that made it extreme matters only through which readings are least
far away.